<a href="https://colab.research.google.com/github/mgcprojects/fraud-detection-deep-clustering/blob/main/DNN_Tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# mounting google drive space for data persistence
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

import torch
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset

## Basic Config
SEED = 42 # Fixed seed for reproducibility of random operations
random.seed(SEED) # Set seed for Python's built-in random module
np.random.seed(SEED) # Set seed for NumPy's random number generation
torch.manual_seed(SEED) # Set seed for PyTorch's CPU random number generation
DATA_PATH = "/content/drive/MyDrive/data/fraud_detection/creditcard.csv" # Set path for credit card csv model data

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Use GPU if available or CPU otherwise

LATENT_SIZE = 32 # Dimensionality of the autoencoder's latent space
MIN_CLUSTER_SIZE = 10 # Min number of samples for a cluster
AE_EPOCHS = 25 # Max number of epochs for training the autoencoder
CLF_EPOCHS = 20 # Max number of epochs for training the classifiers
BATCH_SIZE = 512 # Number of samples per batch during training
EARLY_PATIENCE = 5 # Number of epochs to wait for improvement before stopping training
TRAIN_SPLIT, VAL_SPLIT = 0.6, 0.2 # Proportions for train and validation splits. Test split is inferred.
K_LIST = [2, 4, 5, 10, 20, 50] # List of k-values, clusters to experiment with

## Early Stopper adds a simple class to implement early stopping during model training
class EarlyStopper:
    def __init__(self, patience=EARLY_PATIENCE): # Initializes the early stopper
        self.patience = patience # Number of epochs to wait for improvement
        self.lowest = np.inf # Initialize lowest validation loss to infinity
        self.best_state = None # Stores the state dictionary of the best model.
        self.wait = 0 # Counter for epochs without improvement.

    def step(self, loss, model): # Checks if the validation loss has improved and updates the patience counter
        if loss < self.lowest: # If current loss is better than the best so far...
            self.lowest = loss # ...update the best loss.
            self.best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()} # Save the model's state
            self.wait = 0 # Reset the patience counter
            return False # Indicate that training should continue
        else: # If loss has not improved...
            self.wait += 1 # ...increment the patience counter
            return self.wait >= self.patience # Return True to stop if patience is exceeded

    def restore(self, model): # Restores the model to its best-performing state
        if self.best_state: # Check if a best state was saved
            model.load_state_dict(self.best_state) # Load the best model weights

## Network Architectures
class EncoderAuto(nn.Module): # Compact autoencoder for nonlinear feature embedding
    def __init__(self, n_features, latent_dim=LATENT_SIZE): # Define the autoencoder architecture
        super().__init__()
        self.encoder = nn.Sequential( # The encoder part compresses input to the latent space
            nn.Linear(n_features, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, latent_dim)
        )
        self.decoder = nn.Sequential( # The decoder part reconstructs the input from the latent space
            nn.Linear(latent_dim, 64), nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, n_features)
        )

    def forward(self, x): # Defines the forward pass for the autoencoder
        z = self.encoder(x) # Pass input through the encoder to get the latent representation.
        out = self.decoder(z) # Pass latent representation through the decoder to get the reconstruction.
        return out, z # Return both the reconstructed output and the latent vector.

class FraudNet(nn.Module): # Simple MLP classifier used per cluster
    def __init__(self, n_inputs): # Defines the MLP classifier architecture
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_inputs, 64), nn.ReLU(), # First hidden layer with ReLU activation
            nn.Dropout(0.25),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(32, 1)
        )

    def forward(self, x): # Defines the forward pass for the classifier
        return self.net(x).squeeze(-1) # Pass input through the network and remove the last dimension

## Preprocessing
def normalize_data(df): # Normalizes 'Time' and 'Amount' columns in the dataframe
    dfc = df.copy() # Create a copy of the dataframe to avoid modifying the original
    # Feature scaling for numerical columns
    if {'Time', 'Amount'}.issubset(dfc.columns): # Check if 'Time' and 'Amount' columns exist
        dfc['scaled_time'] = StandardScaler().fit_transform(dfc['Time'].values.reshape(-1, 1)) # Scale the 'Time' column
        dfc['scaled_amount'] = StandardScaler().fit_transform(dfc['Amount'].values.reshape(-1, 1)) # Scale the 'Amount' column
        dfc = dfc.drop(columns=['Time', 'Amount']) # Drop the original unscaled columns
    else: # Handle cases where one or both columns might be missing
        if 'Time' in dfc.columns: # If only 'Time' exists...
            dfc = dfc.drop(columns=['Time']) # ...drop it
    return dfc # Return the preprocessed dataframe

def split_dataset(df, stratified=False): # Split dataset into train, val, test using time or stratified sampling
    if not stratified and 'scaled_time' in df.columns: # Use temporal split if not stratified and time is available
        # Approximation to temporal split using time column order
        df_sorted = df.sort_values(by='scaled_time').reset_index(drop=True) # Sort data by the scaled time
        n = len(df_sorted) # Get the total number of samples
        t1, t2 = int(n * TRAIN_SPLIT), int(n * (TRAIN_SPLIT + VAL_SPLIT)) # Calculate split points
        train, val, test = df_sorted[:t1], df_sorted[t1:t2], df_sorted[t2:] # Slice the dataframe into train, val, test sets
    else: # Use stratified split as a fallback or if explicitly requested
        X = df.drop(columns=['Class']) # Separate features from the target variable
        y = df['Class'] # Get the target variable
        # First split: create training set and a temporary set for validation and tes.
        X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, train_size=TRAIN_SPLIT, stratify=y, random_state=SEED)
        # Calculate the proportion of the validation set relative to the temporary set
        rel_val = VAL_SPLIT / (1 - TRAIN_SPLIT)
        # Second split: create validation and test sets from the temporary set
        X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, train_size=rel_val, stratify=y_tmp, random_state=SEED)
        # Recombine features and labels into dataframes for consistency
        train = pd.concat([X_train, y_train], axis=1)
        val = pd.concat([X_val, y_val], axis=1)
        test = pd.concat([X_test, y_test], axis=1)
    return train, val, test # Return the three dataframes

## Training for autoencoder and return latent embeddings
def train_autoencoder(X_train):# Trains the autoencoder on the training data to learn feature representations
    model = EncoderAuto(X_train.shape[1]).to(DEVICE) # Initialize the autoencoder model and move it to the selected device
    opt = optim.Adam(model.parameters(), lr=1e-3) # Use the Adam optimizer
    loss_fn = nn.MSELoss() # Use Mean Squared Error for reconstruction loss

    # Create a DataLoader to handle batching and shuffling of training data
    loader = DataLoader(TensorDataset(torch.tensor(X_train, dtype=torch.float32)), batch_size=BATCH_SIZE, shuffle=True)
    stopper = EarlyStopper() # Initialize the early stopping mechanism

    for ep in range(AE_EPOCHS): # Loop over the specified number of epochs
        total = 0.0 # Initialize total loss for the epoch
        for (xb,) in loader: # Iterate over batches of data
            xb = xb.to(DEVICE) # Move the batch to the selected device
            opt.zero_grad() # Clear previous gradients
            out, _ = model(xb) # Forward pass to get reconstruction and latent vector
            loss = loss_fn(out, xb) # Calculate the reconstruction loss
            loss.backward() # Backward pass to compute gradients
            opt.step() # Update model weights
            total += loss.item() * xb.size(0) # Accumulate the batch loss
        avg_loss = total / len(loader.dataset) # Calculate the average loss for the epoch
        if stopper.step(avg_loss, model): # Check for early stopping condition
            break # Exit the training loop if stopping is triggered

    stopper.restore(model) # Restore the model to its best state
    model.eval() # Set the model to evaluation mode
    return model # Return the trained autoencoder

def get_latent(model, X): # Encode features to latent representation
    # Uses the trained autoencoder's encoder part to transform data into the latent space
    model.eval() # Ensure the model is in evaluation mode
    Z = [] # Initialize a list to store latent vectors
    with torch.no_grad(): # Disable gradient calculation for inference
        for i in range(0, len(X), 2048): # Process the data in batches to manage memory
            xb = torch.tensor(X[i:i+2048], dtype=torch.float32).to(DEVICE) # Convert batch to tensor and move to device
            _, z = model(xb) # Get the latent representation from the autoencoder
            Z.append(z.cpu().numpy()) # Move latent vectors to CPU, convert to NumPy, and append to the list
    return np.vstack(Z) # Stack all batch results into a single NumPy array

def train_cluster_classifier(X, y, Xv, yv): # Train an MLP classifier for a single cluster

    # Trains a dedicated MLP classifier for a specific data cluster
    model = FraudNet(X.shape[1]).to(DEVICE) # Initialize the MLP classifier and move to device
    opt = optim.Adam(model.parameters(), lr=1e-3) # Use the Adam optimizer

    # Handle imbalance
    pos = max(y.sum(), 1) # Count positive fraud samples, ensuring it's at least 1 to avoid division by zero
    neg = max(len(y) - pos, 1) # Count negative non-fraud samples
    pos_weight = torch.tensor([neg / pos], dtype=torch.float32).to(DEVICE) # Calculate weight for the positive class to counteract imbalance
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight) # Use Binary Cross-Entropy with logits applying the class weight

    # Create a dataset and dataloader for the cluster's training data
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32))
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True)
    stopper = EarlyStopper() # Initialize early stopping

    for epoch in range(CLF_EPOCHS): # Loop over the specified number of epochs
        model.train() # Set the model to training mode
        for xb, yb in loader: # Iterate over batches
            xb, yb = xb.to(DEVICE), yb.to(DEVICE) # Move data to the device
            opt.zero_grad() # Clear gradients
            logits = model(xb) # Get model predictions (logits)
            loss = loss_fn(logits, yb) # Calculate the loss
            loss.backward() # Compute gradients
            opt.step() # Update weights

        # simple val check
        if len(Xv) > 0: # Perform validation if validation data is available
            model.eval() # Set model to evaluation mode
            with torch.no_grad(): # Disable gradient calculation
                # Get predictions for the validation set
                val_logits = model(torch.tensor(Xv, dtype=torch.float32).to(DEVICE))
                # Calculate validation loss
                val_loss = loss_fn(val_logits, torch.tensor(yv, dtype=torch.float32).to(DEVICE)).item()
            if stopper.step(val_loss, model): # Check for early stopping
                break # Stop training if triggered
    stopper.restore(model) # Restore the best model weights
    return model # Return the trained classifier

## Evaluation for the performance of the cluster-based model approach on the test set
def evaluate_clusters(Z_test, y_test, kmeans, cluster_models, backup_model):
    preds = np.zeros(len(Z_test)) # Initialize an array to store prediction probabilities
    centers = kmeans.cluster_centers_ # Get the coordinates of the cluster centroids
    for i, z in enumerate(Z_test): # Iterate through each test sample's latent vector

        # Find the closest cluster centroid for the current sample
        cluster = np.argmin(np.linalg.norm(centers - z, axis=1))

        # Select the appropriate model for the assigned cluster, or the fallback model
        model = cluster_models.get(cluster, backup_model)
        with torch.no_grad(): # Perform inference without gradient calculation
            x = torch.tensor(z, dtype=torch.float32).to(DEVICE) # Convert sample to a tensor
            logit = model(x).cpu().numpy().ravel() # Get the raw logit prediction
            prob = 1 / (1 + np.exp(-logit)) # Convert logit to a probability using the sigmoid function
        preds[i] = prob.item() # Store the prediction probability
    labels = (preds >= 0.5).astype(int) # Convert probabilities to binary class labels 0,1

    # Calculate and store various performance metrics
    metrics = {
        "Precision": precision_score(y_test, labels, zero_division=0), # Calculate precision
        "Recall": recall_score(y_test, labels, zero_division=0), # Calculate recall
        "F1": f1_score(y_test, labels, zero_division=0), # Calculate F1-score
        "ROC_AUC": roc_auc_score(y_test, preds), # Calculate ROC AUC score using probabilities
        "PR_AUC": average_precision_score(y_test, preds), # Calculate Precision-Recall AUC
        "Confusion": confusion_matrix(y_test, labels) # Generate the confusion matrix
    }
    return metrics # Return the dictionary of metrics

## Main function to execute the entire deep clustering pipeline
def main(data_path=DATA_PATH, stratified=False, ks=K_LIST):
    print(f"Device: {DEVICE}") # Print the device being used either CPU or GPU
    data = pd.read_csv(data_path) # Load the dataset
    df = normalize_data(data) # Preprocess the data
    train, val, test = split_dataset(df, stratified=stratified) # Split data into train, validation, and test sets

    # Drop rows with NaN in 'Class' from the test set before extracting y_test
    test = test.dropna(subset=['Class']) # Clean the test set by removing rows with missing target values


    features = [c for c in df.columns if c != "Class"] # Define the list of feature columns
    scaler = StandardScaler() # Initialize a standard scaler for feature normalization
    X_train = scaler.fit_transform(train[features]) # Fit the scaler on training data and transform it
    X_val = scaler.transform(val[features]) # Transform validation data using the fitted scaler
    # Ensure X_test aligns with the cleaned test DataFrame
    X_test = scaler.transform(test[features]) # Transform the clean test data

    y_train = train["Class"].values # Extract target labels for the training set
    y_val = val["Class"].values # Extract target labels for the validation set
    y_test = test["Class"].values # Extract target labels for the test set

    print(f"Training autoencoder on {X_train.shape[0]} samples...") # Log the start of autoencoder training
    ae = train_autoencoder(X_train) # Train the autoencoder on the scaled training features
    Z_train = get_latent(ae, X_train) # Generate latent representations for the training set
    Z_val = get_latent(ae, X_val) # Generate latent representations for the validation set
    Z_test = get_latent(ae, X_test) # Generate latent representations for the test set

    results = [] # Initialize a list to store results for each k value
    for k in ks: # Loop through the list of k values to test
        print(f"\n--- Clustering with k={k} ---") # Log the current k value
        # Use MiniBatchKMeans for large datasets for efficiency, otherwise use standard KMeans
        km = MiniBatchKMeans(n_clusters=k, random_state=SEED, batch_size=1024, n_init=10) if len(Z_train) > 20000 else KMeans(n_clusters=k, random_state=SEED, n_init=10)
        km.fit(Z_train) # Fit the KMeans model on the latent training data

        cluster_models = {} # Dictionary to hold the trained model for each cluster
        cluster_ids = km.predict(Z_train) # Assign each training sample to a cluster
        val_ids = km.predict(Z_val) # Assign each validation sample to a cluster

        for cid in range(k): # Iterate through each cluster ID
            idx = np.where(cluster_ids == cid)[0] # Get indices of training samples in the current cluster
            vdx = np.where(val_ids == cid)[0] # Get indices of validation samples in the current cluster
            if len(idx) < MIN_CLUSTER_SIZE: # If a cluster is too small...
                cluster_models[cid] = None # ...skip training a specific model for it
                continue
            Xc, yc = Z_train[idx], y_train[idx] # Get the training data for the cluster
            Xv, yv = Z_val[vdx], y_val[vdx] # Get the validation data for the cluster
            print(f"Cluster {cid}: {len(idx)} samples, fraud={int(yc.sum())}") # Log cluster size and fraud count
            model = train_cluster_classifier(Xc, yc, Xv, yv) # Train a classifier for this specific cluster
            cluster_models[cid] = model # Store the trained model

        print("Training fallback model for small and empty clusters") # Log fallback model training
        # Train a global model on all training data to use for small or unassigned clusters
        fallback = train_cluster_classifier(Z_train, y_train, Z_val, y_val)

        # Evaluate the performance of the cluster-based approach on the test set
        metrics = evaluate_clusters(Z_test, y_test, km, cluster_models, fallback)
        results.append((k, metrics)) # Store the metrics for the current k
        print(f"Results (k={k}): {metrics}") # Print the results

    # Export summary
    dfres = pd.DataFrame([{ # Create a pandas DataFrame from the collected results
        "k": k,
        "Precision": m["Precision"],
        "Recall": m["Recall"],
        "F1": m["F1"],
        "ROC_AUC": m["ROC_AUC"],
        "PR_AUC": m["PR_AUC"]
    } for k, m in results])
    dfres.to_csv("test1_results.csv", index=False) # Save the summary DataFrame to a CSV file
    print("\nSaved metrics to test1_results.csv") # Confirm that the results were saved
    return dfres # Return the results DataFrame

if __name__ == "__main__":
    main() # Excecute main function

Device: cpu
Training autoencoder on 170884 samples...


KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Report on RNN Model over Apple Stock Price

The model with LSTM architecture, moderate dropout regularization, and light L2 weight decay shows good performance and generalization. The model achieves strong training metrics (MAE of \$4.97, R2 of 0.9878) while maintaining reasonable test results with MAE of \$15.80, R2 of 0.5953, and MAPE of 7.27%. This represents a 44% improvement over the previous iteration and importantly achieves a positive test R2, meaning the model explains about 60% of the variance in unseen data. The overfitting ratio (test MAE divided by train MAE) is 3.2, indicating there is still some memorization but it is much more controlled than in earlier models. The MAPE of 7.27% indicates that on average, predictions deviate by about \$15-16 for stocks in the $200 range, which is approaching practical usefulness.

 To further improve the model, several strategies could be explored:
 - Implementing ensemble methods (combining several models or using boosting) to reduce variance
 - Predicting percentage returns rather than absolute prices to better handle non-stationarity
 - Adding more features such as trading volume, volatility indices, and macroeconomic data
 - Using more advanced architectures such as Transformer models with attention mechanisms
 - Using time-series cross-validation for better real-world performance assessment

Ultimately, stock prices may be very hard to predict because of market efficiency, which suggests the problem might be better framed as classifying trends (up, down, or sideways) rather than predicting exact prices. Predicting stock prices is a challenging problem because it requires a large amount of context and information to make accurate decisions at a fast pace.